# Notebook 01 — Data Cleaning & Merging

## Overview

This notebook loads all 7 raw CSVs from the League of Legends ranked match dataset, inspects each one individually, executes a multi-step merge pipeline, cleans the result, and outputs a single unified `clean_matches.csv` for downstream EDA and modelling.

### Dataset files

| File | Description | Approx. rows |
|------|-------------|-------------|
| `matches.csv` | One row per match — game ID, duration (seconds), patch version | ~18k |
| `participants.csv` | One row per player per match — links match ↔ champion ↔ player slot | ~184k |
| `stats1.csv` | Combat and economy stats (first half of participants) | ~92k |
| `stats2.csv` | Same columns — second half of participants | ~92k |
| `teamstats.csv` | One row per team per match — objectives, kills | ~36k |
| `teambans.csv` | Champion bans per team per match | ~184k |
| `champs.csv` | Champion ID → name lookup (2016–17 patch) | 138 |
| `League of legend Champions 2024.csv` | Enriched champion metadata (Classes, Role, Difficulty, stats) | 169 |

### Key join path

```
concat(stats1, stats2)           → all participant stats (same schema)
participants.id → stats.id       → player stats
participants.matchid → matches.id → duration, patch
participants.matchid + team_id → teamstats  → objectives
participants.championid → champs.id → Champions 2024.Name  → enrichment
```

### Important notes
- `duration` is in **seconds**, not minutes. All per-minute features must divide by `duration / 60`.
- Matches with `duration < 300s` are **remakes** and will be filtered out.
- `stats1` and `stats2` are **row-wise halves** of the same participant set — same columns, different rows. Concatenate them before joining to participants.
- The 2024 champion dataset covers a different patch era; some old names won't match — handled by name normalization + fallback to 'Unknown'.

## 0. Imports & Paths

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

DATA_DIR  = '../datasets/League of Legends Ranked Matches'
CHAMP_DIR = '../datasets/League of Legends Champion'
OUTPUT_DIR = '../data/processed'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output will be saved to: {os.path.abspath(OUTPUT_DIR)}")

Output will be saved to: C:\Users\asus\rift-analytics\data\processed


## 1. Load Raw Data

We load each CSV individually and report its shape and null counts so we know exactly what we are working with before any joins.

In [2]:
# ── Core match tables ──────────────────────────────────────────────────────────
matches      = pd.read_csv(f'{DATA_DIR}/matches.csv')
participants = pd.read_csv(f'{DATA_DIR}/participants.csv')
stats1       = pd.read_csv(f'{DATA_DIR}/stats1.csv')
stats2       = pd.read_csv(f'{DATA_DIR}/stats2.csv', low_memory=False)
teamstats    = pd.read_csv(f'{DATA_DIR}/teamstats.csv')
teambans     = pd.read_csv(f'{DATA_DIR}/teambans.csv')

# ── Champion lookup tables ─────────────────────────────────────────────────────
champs_old = pd.read_csv(f'{DATA_DIR}/champs.csv')
champs_new = pd.read_csv(f'{CHAMP_DIR}/League of legend Champions 2024.csv')

print("All files loaded successfully.")

All files loaded successfully.


### 1.1 Per-table inspection

For each table we print:
- Shape (rows × columns)
- Column names
- Non-zero null counts
- First 2 rows

In [3]:
def inspect(name, df):
    """Pretty-print shape, columns, nulls and head for a dataframe."""
    print(f"{'='*60}")
    print(f" TABLE : {name}")
    print(f" Shape : {df.shape[0]:,} rows x {df.shape[1]} columns")
    print(f" Dtypes: {df.dtypes.value_counts().to_dict()}")
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls):
        print(" Nulls :")
        for col, n in nulls.items():
            print(f"   {col}: {n:,}")
    else:
        print(" Nulls : None")
    print(f"{'='*60}")
    display(df.head(2))
    print()

In [4]:
inspect('matches', matches)

 TABLE : matches
 Shape : 184,069 rows x 8 columns
 Dtypes: {dtype('int64'): 6, <StringDtype(na_value=nan)>: 2}
 Nulls : None


,id,gameid,platformid,queueid,seasonid,duration,creation,version
0,10,3187427022,EUW1,420,8,1909,1495068946860,7.10.187.9675
1,11,3187425281,EUW1,420,8,1693,1495066760778,7.10.187.9675


In [5]:
inspect('participants', participants)

 TABLE : participants
 Shape : 1,834,520 rows x 8 columns
 Dtypes: {dtype('int64'): 6, <StringDtype(na_value=nan)>: 2}
 Nulls : None


,id,matchid,player,championid,ss1,ss2,role,position
0,9,10,1,19,4,11,NONE,JUNGLE
1,10,10,2,267,3,4,DUO_SUPPORT,BOT


In [6]:
inspect('stats1', stats1)
print("NOTE: stats1 covers the FIRST half of participant IDs")

 TABLE : stats1
 Shape : 999,999 rows x 56 columns
 Dtypes: {dtype('int64'): 56}
 Nulls : None


,id,win,item1,item2,item3,item4,item5,item6,trinket,kills,...,neutralminionskilled,ownjunglekills,enemyjunglekills,totcctimedealt,champlvl,pinksbought,wardsbought,wardsplaced,wardskilled,firstblood
0,9,0,3748,2003,3111,3053,1419,1042,3340,6,...,69,42,27,610,13,0,0,10,0,0
1,10,0,2301,3111,3190,3107,0,0,3364,0,...,1,1,0,211,14,1,0,17,3,0



NOTE: stats1 covers the FIRST half of participant IDs


In [7]:
inspect('stats2', stats2)
print("NOTE: stats2 covers the SECOND half of participant IDs — same schema as stats1")

 TABLE : stats2
 Shape : 834,518 rows x 56 columns
 Dtypes: {dtype('int64'): 55, <StringDtype(na_value=nan)>: 1}


 Nulls : None


,id,win,item1,item2,item3,item4,item5,item6,trinket,kills,...,neutralminionskilled,ownjunglekills,enemyjunglekills,totcctimedealt,champlvl,pinksbought,wardsbought,wardsplaced,wardskilled,firstblood
0,1028382,0,1056,3001,1052,3020,1058,1026,3340,7,...,0,0,0,50,12,0,0,6,0,0
1,1028383,0,1041,2003,0,0,0,0,3340,0,...,11,11,0,114,3,0,0,0,0,0



NOTE: stats2 covers the SECOND half of participant IDs — same schema as stats1


In [8]:
inspect('teamstats', teamstats)

 TABLE : teamstats
 Shape : 368,138 rows x 13 columns
 Dtypes: {dtype('int64'): 13}
 Nulls : None


,matchid,teamid,firstblood,firsttower,firstinhib,firstbaron,firstdragon,firstharry,towerkills,inhibkills,baronkills,dragonkills,harrykills
0,10,100,0,1,0,0,0,0,5,0,0,0,0
1,10,200,1,0,1,1,1,1,10,3,1,3,1


In [9]:
inspect('teambans', teambans)

 TABLE : teambans
 Shape : 1,099,185 rows x 4 columns
 Dtypes: {dtype('int64'): 4}


 Nulls : None

,matchid,teamid,championid,banturn
0,10,100,11,1
1,10,100,117,3


In [10]:
inspect('champs_old (2016/17)', champs_old)
print("\n")
inspect('champs_new (2024)', champs_new)

 TABLE : champs_old (2016/17)
 Shape : 138 rows x 2 columns
 Dtypes: {<StringDtype(na_value=nan)>: 1, dtype('int64'): 1}
 Nulls : None


,name,id
0,Jax,24
1,Sona,37





 TABLE : champs_new (2024)
 Shape : 168 rows x 13 columns
 Dtypes: {<StringDtype(na_value=nan)>: 9, dtype('int64'): 4}
 Nulls : None


,Name,Nick Name,Classes,Release Date,Last Changed,Blue Essence,RP,Difficulty,Role,Range type,Resourse type,Base HP,Base mana
0,Aatrox,The darkin blade,Juggernaut,2013-06-13,V14.14,4800,880,Advanced,Top,Melee,Blood Well,650,0
1,Ahri,The nine-tailed fox,Burst,2011-12-14,V14.18,3150,790,Intermediate,Middle,Ranged,Mana,590,418


## 2. Prepare Champion Metadata

The old dataset uses internal API names (no spaces, no accents — e.g. 'MonkeyKing' instead of 'Wukong').  
The 2024 dataset uses display names. We normalize both to lowercase with punctuation stripped, then apply known manual corrections before joining.

In [11]:
def clean_champ_name(name):
    """Lowercase, strip spaces / apostrophes / dots for fuzzy matching."""
    if not isinstance(name, str):
        return name
    return name.lower().replace(" ", "").replace("'", "").replace(".", "")

champs_old['name_clean'] = champs_old['name'].apply(clean_champ_name)
champs_new['name_clean'] = champs_new['Name'].apply(clean_champ_name)

print("Sample cleaned champion names (old dataset):")
print(champs_old[['name','name_clean']].head(12).to_string(index=False))

Sample cleaned champion names (old dataset):
      name name_clean
       Jax        jax
      Sona       sona
  Tristana   tristana
     Varus      varus
     Fiora      fiora
    Singed     singed
Tahm Kench  tahmkench
   LeBlanc    leblanc
    Thresh     thresh
     Karma      karma
      Jhin       jhin
    Rumble     rumble


In [12]:
# ── Find mismatches BEFORE manual corrections ──────────────────────────────────
old_names = set(champs_old['name_clean'])
new_names = set(champs_new['name_clean'])
unmatched_before = old_names - new_names

print(f"Champions in old dataset : {len(old_names)}")
print(f"Champions in new dataset : {len(new_names)}")
print(f"Unmatched before corrections: {len(unmatched_before)}")
print("Unmatched names:")
for n in sorted(unmatched_before):
    # Show the original raw name
    orig = champs_old.loc[champs_old['name_clean'] == n, 'name'].values
    print(f"  clean='{n}'  raw='{orig[0] if len(orig) else n}'")

Champions in old dataset : 138
Champions in new dataset : 168
Unmatched before corrections: 1
Unmatched names:
  clean='nunu'  raw='Nunu'


In [13]:
# ── Manual corrections for known Riot API name → display name mismatches ───────
corrections = {
    'MonkeyKing'  : 'wukong',
    'ChoGath'     : 'chogath',
    'KhaZix'      : 'khazix',
    'Leblanc'     : 'leblanc',
    'VelKoz'      : 'velkoz',
    'RekSai'      : 'reksai',
    'AurelionSol' : 'aurelionsol',
    'TahmKench'   : 'tahmkench',
}

for raw_name, corrected_clean in corrections.items():
    mask = champs_old['name'] == raw_name
    if mask.any():
        champs_old.loc[mask, 'name_clean'] = corrected_clean
        print(f"  Corrected: '{raw_name}' -> '{corrected_clean}'")

# Recheck
old_names_after = set(champs_old['name_clean'])
still_unmatched = old_names_after - new_names
print(f"\nAfter corrections — still unmatched: {len(still_unmatched)}")
if still_unmatched:
    print("These will get 'Unknown' for Classes/Role/Difficulty:")
    for n in sorted(still_unmatched):
        print(f"  '{n}'")

  Corrected: 'ChoGath' -> 'chogath'
  Corrected: 'KhaZix' -> 'khazix'
  Corrected: 'VelKoz' -> 'velkoz'
  Corrected: 'RekSai' -> 'reksai'

After corrections — still unmatched: 1
These will get 'Unknown' for Classes/Role/Difficulty:
  'nunu'


In [14]:
# ── Join old champion IDs to 2024 enriched metadata ───────────────────────────
champ_meta = champs_old.merge(
    champs_new[['name_clean', 'Classes', 'Role', 'Difficulty']],
    on='name_clean',
    how='left'
)

# Fill missing metadata gracefully (no ChainedAssignment)
champ_meta['Classes']    = champ_meta['Classes'].fillna('Unknown')
champ_meta['Role']       = champ_meta['Role'].fillna('Unknown')
champ_meta['Difficulty'] = champ_meta['Difficulty'].fillna('Unknown')

print(f"Champion metadata shape        : {champ_meta.shape}")
print(f"Champions with class data      : {(champ_meta['Classes'] != 'Unknown').sum()}  / {len(champ_meta)}")
print(f"Champions mapped to 'Unknown'  : {(champ_meta['Classes'] == 'Unknown').sum()}")
print()
display(champ_meta[['id','name','Classes','Role','Difficulty']].head(10))

Champion metadata shape        : (138, 6)
Champions with class data      : 137  / 138
Champions mapped to 'Unknown'  : 1



,id,name,Classes,Role,Difficulty
0,24,Jax,Skirmisher,"Top,Jungle",Novice
1,37,Sona,Enchanter,Support,Novice
2,18,Tristana,Marksman,"Bottom,Middle",Beginner
3,110,Varus,Marksman Artillery,Bottom,Intermediate
4,114,Fiora,Skirmisher,Top,Intermediate
5,27,Singed,Specialist,Top,Intermediate_Plus
6,223,Tahm Kench,Warden,"Top,Support",Beginner
7,7,LeBlanc,Burst,Middle,Advanced
8,412,Thresh,Catcher,Support,Advanced
9,43,Karma,Burst Enchanter,Jungle,Novice


## 3. Merge Pipeline

We build the master dataframe step-by-step:

```
1. concat(stats1, stats2)           → full stats table (row-wise)
2. participants JOIN stats ON id    → player-level table
3. result JOIN matches ON matchid   → add duration, timestamp
4. derive team_id (player 1-5 = 100, 6-10 = 200)
5. result JOIN teamstats ON (matchid, team_id) → objectives
6. result JOIN champ_meta ON championid → champion enrichment
```

In [15]:
# Step 1: stats1 and stats2 are ROW-WISE halves of the same participant set
# They share the exact same columns — concatenate them to get the full stats table
print(f"stats1 shape : {stats1.shape}")
print(f"stats2 shape : {stats2.shape}")
print(f"ID overlap   : {len(set(stats1['id']) & set(stats2['id']))} (should be 0)")

stats = pd.concat([stats1, stats2], ignore_index=True)
print(f"Combined stats shape: {stats.shape}")

stats1 shape : (999999, 56)
stats2 shape : (834518, 56)


ID overlap   : 0 (should be 0)


Combined stats shape: (1834517, 56)


In [16]:
# Step 2: Join participants with their stats
df = pd.merge(participants, stats, on='id', how='left')
print(f"After join stats    : {df.shape}")

After join stats    : (1834520, 63)


In [17]:
# Step 3: Join matches to get duration and timestamp
df = pd.merge(
    df,
    matches[['id', 'duration', 'creation']],
    left_on='matchid', right_on='id',
    suffixes=('', '_match'),
    how='left'
)
# Drop duplicate 'id' column from matches
if 'id_match' in df.columns:
    df.drop('id_match', axis=1, inplace=True)

# Step 4: Derive team_id from player slot (1-5 = team 100, 6-10 = team 200)
df['team_id'] = np.where(df['player'] <= 5, 100, 200)

print(f"After join matches + team_id: {df.shape}")

After join matches + team_id: (1834520, 66)


In [18]:
# Step 5: Join teamstats to get objectives
ts_columns = [
    'matchid', 'teamid',
    'firstblood', 'firsttower', 'firstinhib',
    'firstbaron', 'firstdragon', 'firstharry',
    'towerkills', 'inhibkills', 'baronkills', 'dragonkills', 'harrykills'
]
ts_columns = [c for c in ts_columns if c in teamstats.columns]

df = pd.merge(
    df,
    teamstats[ts_columns],
    left_on=['matchid', 'team_id'],
    right_on=['matchid', 'teamid'],
    how='left'
)
if 'teamid' in df.columns:
    df.drop('teamid', axis=1, inplace=True)

print(f"After join teamstats: {df.shape}")

After join teamstats: (1834520, 77)


In [19]:
# Step 6: Join champion metadata
df = pd.merge(
    df,
    champ_meta[['id', 'name', 'Classes', 'Role', 'Difficulty']],
    left_on='championid', right_on='id',
    suffixes=('', '_champ'),
    how='left'
)
df.rename(columns={
    'name'      : 'champion_name',
    'Classes'   : 'champion_classes',
    'Role'      : 'champion_role',
    'Difficulty': 'champion_difficulty'
}, inplace=True)

if 'id_champ' in df.columns:
    df.drop('id_champ', axis=1, inplace=True)

# Standardize ALL column names to lowercase
df.columns = [col.lower() for col in df.columns]

print(f"After join champ_meta : {df.shape}")
print(f"All columns: {list(df.columns)}")

After join champ_meta : (1834520, 81)
All columns: ['id', 'matchid', 'player', 'championid', 'ss1', 'ss2', 'role', 'position', 'win', 'item1', 'item2', 'item3', 'item4', 'item5', 'item6', 'trinket', 'kills', 'deaths', 'assists', 'largestkillingspree', 'largestmultikill', 'killingsprees', 'longesttimespentliving', 'doublekills', 'triplekills', 'quadrakills', 'pentakills', 'legendarykills', 'totdmgdealt', 'magicdmgdealt', 'physicaldmgdealt', 'truedmgdealt', 'largestcrit', 'totdmgtochamp', 'magicdmgtochamp', 'physdmgtochamp', 'truedmgtochamp', 'totheal', 'totunitshealed', 'dmgselfmit', 'dmgtoobj', 'dmgtoturrets', 'visionscore', 'timecc', 'totdmgtaken', 'magicdmgtaken', 'physdmgtaken', 'truedmgtaken', 'goldearned', 'goldspent', 'turretkills', 'inhibkills_x', 'totminionskilled', 'neutralminionskilled', 'ownjunglekills', 'enemyjunglekills', 'totcctimedealt', 'champlvl', 'pinksbought', 'wardsbought', 'wardsplaced', 'wardskilled', 'firstblood_x', 'duration', 'creation', 'team_id', 'firstblood_

## 4. Filtering & Outlier Analysis

### 4.1 Duration distribution & remake filtering

Matches shorter than 300 seconds (5 minutes) are **remakes** — both teams agree to surrender before the game properly begins. Their stats are meaningless and would heavily skew all per-minute features.

In [20]:
duration_min = df['duration'] / 60
print("Duration distribution (minutes):")
print(duration_min.describe().round(2))
print(f"\nBreakdown by duration bucket:")
print(f"  < 5 min  (remakes)  : {(df['duration'] < 300).sum():>10,}")
print(f"  5-10 min            : {((df['duration'] >= 300) & (df['duration'] < 600)).sum():>10,}")
print(f"  10-20 min           : {((df['duration'] >= 600) & (df['duration'] < 1200)).sum():>10,}")
print(f"  20-40 min           : {((df['duration'] >= 1200) & (df['duration'] < 2400)).sum():>10,}")
print(f"  > 40 min            : {(df['duration'] >= 2400).sum():>10,}")

Duration distribution (minutes):
count    1834520.00
mean          30.56
std            8.49
min            3.17
25%           25.70
50%           30.63
75%           35.75
max           83.18
Name: duration, dtype: float64

Breakdown by duration bucket:
  < 5 min  (remakes)  :     47,952
  5-10 min            :        606
  10-20 min           :     61,230
  20-40 min           :  1,511,590
  > 40 min            :    213,142


In [21]:
# Filter out remakes (duration < 5 minutes)
before = len(df)
df = df[df['duration'] >= 300].copy()
after = len(df)
print(f"Removed {before - after:,} remake rows  ({(before-after)/before*100:.2f}% of total)")
print(f"Remaining rows: {after:,}")

Removed 47,952 remake rows  (2.61% of total)
Remaining rows: 1,786,568


### 4.2 Outlier Analysis — Kills, Deaths, Assists

We inspect the extreme tail of combat stats before applying any caps. This ensures the cap thresholds are evidence-based.

In [22]:
for col in ['kills', 'deaths', 'assists']:
    print(f"\n── {col.upper()} ──")
    print(df[col].describe().round(2))
    print(f"  99th pct  : {df[col].quantile(0.99):.1f}")
    print(f"  99.9th pct: {df[col].quantile(0.999):.1f}")
    print(f"  Max value : {df[col].max():.0f}")
    print(f"  Rows > 20 : {(df[col] > 20).sum():,}")
    print(f"  Rows > 30 : {(df[col] > 30).sum():,}")


── KILLS ──


count    1786565.00
mean           5.94
std            4.57
min            0.00
25%            2.00
50%            5.00
75%            9.00
max           45.00
Name: kills, dtype: float64


  99th pct  : 20.0
  99.9th pct: 26.0
  Max value : 45
  Rows > 20 : 14,113
  Rows > 30 : 374

── DEATHS ──


count    1786565.00
mean           5.97
std            3.17
min            0.00
25%            4.00
50%            6.00
75%            8.00
max           38.00
Name: deaths, dtype: float64


  99th pct  : 14.0
  99.9th pct: 18.0
  Max value : 38
  Rows > 20 : 238
  Rows > 30 : 4

── ASSISTS ──


count    1786565.00
mean           8.54
std            5.84
min            0.00
25%            4.00
50%            7.00
75%           12.00
max           57.00
Name: assists, dtype: float64
  99th pct  : 27.0
  99.9th pct: 34.0
  Max value : 57
  Rows > 20 : 77,145
  Rows > 30 : 5,890


In [23]:
# Cap at 99.9th percentile to remove extreme outliers while preserving the long tail
for col in ['kills', 'deaths', 'assists']:
    cap = df[col].quantile(0.999)
    n_capped = (df[col] > cap).sum()
    df[col] = df[col].clip(upper=cap)
    print(f"{col:10} capped at {cap:.0f}  ({n_capped:,} rows affected)")

kills      capped at 26  (1,647 rows affected)
deaths     capped at 18  (929 rows affected)


assists    capped at 34  (1,700 rows affected)


### 4.3 Missing Values Analysis

In [24]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

if len(missing) > 0:
    print("Columns with missing values:")
    print(missing.to_string())
else:
    print("No missing values.")

Columns with missing values:
win                       3
item1                     3
item2                     3
item3                     3
item4                     3
item5                     3
item6                     3
trinket                   3
kills                     3
deaths                    3
assists                   3
largestkillingspree       3
largestmultikill          3
killingsprees             3
longesttimespentliving    3
doublekills               3
triplekills               3
quadrakills               3
pentakills                3
legendarykills            3
totdmgdealt               3
magicdmgdealt             3
physicaldmgdealt          3
truedmgdealt              3
largestcrit               3
totdmgtochamp             3
magicdmgtochamp           3
physdmgtochamp            3
truedmgtochamp            3
totheal                   3
totunitshealed            3
dmgselfmit                3
dmgtoobj                  3
dmgtoturrets              3
visionscore        

In [25]:
# Boolean event flags: NaN = event didn't happen for this team -> fill with 0
flag_cols = ['firstblood', 'firsttower', 'firstinhib', 'firstbaron', 'firstdragon', 'firstharry']
flag_cols = [c for c in flag_cols if c in df.columns]
df[flag_cols] = df[flag_cols].fillna(0)

# Champion metadata: already 'Unknown' in champ_meta, just be safe
for col in ['champion_classes', 'champion_role', 'champion_difficulty']:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown')

# Final null check
remaining = df.isnull().sum()
remaining = remaining[remaining > 0]
if len(remaining) > 0:
    print("Remaining nulls after fill:")
    print(remaining.to_string())
else:
    print("All nulls resolved.")

Remaining nulls after fill:
win                       3
item1                     3
item2                     3
item3                     3
item4                     3
item5                     3
item6                     3
trinket                   3
kills                     3
deaths                    3
assists                   3
largestkillingspree       3
largestmultikill          3
killingsprees             3
longesttimespentliving    3
doublekills               3
triplekills               3
quadrakills               3
pentakills                3
legendarykills            3
totdmgdealt               3
magicdmgdealt             3
physicaldmgdealt          3
truedmgdealt              3
largestcrit               3
totdmgtochamp             3
magicdmgtochamp           3
physdmgtochamp            3
truedmgtochamp            3
totheal                   3
totunitshealed            3
dmgselfmit                3
dmgtoobj                  3
dmgtoturrets              3
visionscore         

## 5. Select Final Columns & Save

We keep only the columns needed for EDA, feature engineering, and modelling. This reduces file size significantly without losing any analytical value.

In [26]:
cols_to_keep = [
    # Identity
    'id', 'matchid', 'player',
    # Champion info
    'champion_name', 'champion_classes', 'champion_role', 'champion_difficulty',
    # Role/position in game
    'role', 'position',
    # Core outcome
    'win',
    # Combat stats
    'kills', 'deaths', 'assists',
    'totdmgtochamp', 'totdmgdealt', 'totdmgtaken',
    # Economy
    'goldearned', 'goldspent',
    # CS (creep score)
    'totminionskilled', 'neutralminionskilled',
    # Vision
    'visionscore', 'wardsplaced', 'wardskilled',
    # Match context
    'duration', 'team_id',
    # Team objectives
    'firstblood', 'firsttower', 'firstbaron', 'firstdragon',
    'towerkills', 'dragonkills', 'baronkills',
]

# Only keep columns that actually exist in the merged dataframe
cols_to_keep = [c for c in cols_to_keep if c in df.columns]
df_final = df[cols_to_keep].copy()

print(f"Columns selected : {len(cols_to_keep)}")
print(f"Columns dropped  : {df.shape[1] - len(cols_to_keep)}")
print(f"Final shape      : {df_final.shape}")
print(f"\nFinal columns:")
print(cols_to_keep)

Columns selected : 31
Columns dropped  : 50
Final shape      : (1786568, 31)

Final columns:
['id', 'matchid', 'player', 'champion_name', 'champion_classes', 'champion_role', 'champion_difficulty', 'role', 'position', 'win', 'kills', 'deaths', 'assists', 'totdmgtochamp', 'totdmgdealt', 'totdmgtaken', 'goldearned', 'goldspent', 'totminionskilled', 'neutralminionskilled', 'visionscore', 'wardsplaced', 'wardskilled', 'duration', 'team_id', 'firsttower', 'firstbaron', 'firstdragon', 'towerkills', 'dragonkills', 'baronkills']


In [27]:
output_path = f'{OUTPUT_DIR}/clean_matches.csv'
df_final.to_csv(output_path, index=False)
size_mb = os.path.getsize(output_path) / 1024**2
print(f"Saved  : {os.path.abspath(output_path)}")
print(f"Size   : {size_mb:.1f} MB")
print(f"Rows   : {len(df_final):,}")
print(f"Columns: {df_final.shape[1]}")

Saved  : C:\Users\asus\rift-analytics\data\processed\clean_matches.csv
Size   : 281.5 MB
Rows   : 1,786,568
Columns: 31


## 6. Final Data Quality Summary

In [28]:
print("╔══════════════════════════════════════════════════╗")
print("║        DATA CLEANING — FINAL SUMMARY            ║")
print("╠══════════════════════════════════════════════════╣")
print(f"║  Total rows (player-game pairs) : {len(df_final):>10,} ║")
print(f"║  Total columns                  : {df_final.shape[1]:>10} ║")
print(f"║  Unique matches                 : {df_final['matchid'].nunique():>10,} ║")
print(f"║  Unique champions               : {df_final['champion_name'].nunique():>10,} ║")
print(f"║  Win rate (overall)             : {df_final['win'].mean():>9.1%} ║")
print(f"║  Avg duration (minutes)         : {df_final['duration'].mean()/60:>9.1f} ║")
print(f"║  Champions with class data      : {(df_final['champion_classes'] != 'Unknown').mean():>9.1%} ║")
print(f"║  Remaining nulls                : {df_final.isnull().sum().sum():>10,} ║")
print("╚══════════════════════════════════════════════════╝")

print("\nKey stats summary:")
display(df_final[['kills','deaths','assists','goldearned','visionscore','duration']].describe().round(2))

╔══════════════════════════════════════════════════╗
║        DATA CLEANING — FINAL SUMMARY            ║
╠══════════════════════════════════════════════════╣
║  Total rows (player-game pairs) :  1,786,568 ║
║  Total columns                  :         31 ║
║  Unique matches                 :    179,269 ║
║  Unique champions               :        136 ║
║  Win rate (overall)             :     50.0% ║
║  Avg duration (minutes)         :      31.3 ║


║  Champions with class data      :     99.7% ║
║  Remaining nulls                :         42 ║
╚══════════════════════════════════════════════════╝

Key stats summary:


,kills,deaths,assists,goldearned,visionscore,duration
count,1786565.00,1786565.00,1786565.00,1786565.00,1786565.00,1786568.0
mean,5.94,5.96,8.54,11698.04,14.21,1877.7
std,4.55,3.16,5.83,3667.08,17.48,439.3
min,0.00,0.00,0.00,1070.00,0.00,399.0
25%,2.00,4.00,4.00,9082.00,0.00,1570.0
50%,5.00,6.00,7.00,11473.00,10.00,1851.0
75%,9.00,8.00,12.00,14019.00,22.00,2154.0
max,26.00,18.00,34.00,40982.00,179.00,4991.0
